# Introduction

This notebook will convert model from keras file to tflite.


In [ ]:
import tensorflow as tf
import numpy as np
import os
import numpy as np
import tensorflow as tf
import itertools
print(tf.__version__)
print(np.__version__)

from google.colab import files
# files.download(zip_name)


2.19.0
2.0.2


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
df_train = pd.read_pickle(r'/content/drive/MyDrive/Data/New_VitalDB_2peaks_Onset.pkl')
indices = df_train.index

train_idx, temp_idx = train_test_split(indices, test_size=0.3, random_state=42)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42)

# 2. Extract your full DataFrames using .loc
df_val = df_train.loc[val_idx]
df_train_split = df_train.loc[train_idx]
df_test = df_train.loc[test_idx]
print (df_train)
df_train.to_pickle(r'/content/drive/MyDrive/Test and validation data/Train/VitalDB_Train_Onset.pkl')

                                               Cropped_1s  Peak_Index  \
0       [-10.875439998578193, -11.142804467733681, -11...         563   
1       [-12.414277921858734, -12.301388696403217, -11...         287   
2       [-9.550334092908498, -9.535268964451095, -9.52...         602   
3       [-6.97291877026109, -7.246176236715118, -7.484...         384   
4       [-9.482280671654904, -9.090561717384709, -8.78...         156   
...                                                   ...         ...   
166523  [-10.52681159055055, -10.684670971560417, -10....         809   
166524  [-10.358293662629745, -10.68214189005744, -10....         544   
166525  [-10.115408395251016, -10.433467331586131, -10...         422   
166526  [-13.493630264611479, -13.514172277356975, -13...         108   
166527  [-12.033259830222507, -12.499298357365303, -13...         586   

       Onset_Index  Middle_Index  Case     dt  Segment_Number  Result  \
0       (540, 615)           578     1   3060     

In [ ]:
import pandas as pd
MODEL_DIR = "/content/drive/MyDrive/VPG_tmp_model"
DATASET_PATH = "/content/drive/MyDrive/Test and validation data/Validation/VitalDB_Onset_val.pkl"     # shape: (N, H, W, C) or (N, input_dim)
OUTPUT_DIR = "/content/drive/MyDrive/VPG_tmp_model_quantized"
os.makedirs(OUTPUT_DIR, exist_ok=True)

dataset = pd.read_pickle(DATASET_PATH)
print (dataset)

                                               Cropped_1s  Peak_Index  \
93735   [-8.010889322920201, -8.215599198710054, -8.45...         545   
108932  [-11.18836818668869, -11.53050346909059, -11.8...         162   
153999  [-8.730325506044188, -8.815675212353378, -8.89...          92   
71645   [-10.94706729810931, -11.574720304844696, -12....         376   
113976  [8.80254672151426, 5.724340017805792, 2.867171...         574   
...                                                   ...         ...   
124419  [-9.517770312527912, -9.923431153365106, -10.2...         126   
105995  [-10.073699054138336, -10.173885373544001, -10...         536   
147023  [-8.316489723860094, -7.976102158278096, -7.47...         585   
131125  [-6.559318426148597, -6.537924088720381, -6.52...         361   
148971  [3.814134263058377, 2.1609791225188726, 0.6013...         936   

       Onset_Index  Middle_Index  Case     dt  Segment_Number  Result  \
93735   (527, 613)           570  3783   9415     

In [ ]:
def representative_dataset_gen(rep_data):
    for x in rep_data:
        x = x.astype(np.float32)
        x = x.reshape(1, 98, 1)
        yield [x]
X = np.stack(dataset['APG_Signal_Normalized'].values)

In [ ]:
rep_sizes = [2000, 3000, 4000, 5000]
new_quantizer_list = [True]
new_converter_list = [True]
full_int8_modes = ["strict"]

def convert_model(model_path, rep_data, use_new_quantizer, use_new_converter, int8_mode):
    model = tf.keras.models.load_model(model_path)
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = lambda: representative_dataset_gen(rep_data)

    converter._experimental_new_quantizer = use_new_quantizer
    converter.experimental_new_converter = use_new_converter

    converter.inference_input_type = tf.float32
    converter.inference_output_type = tf.float32

    if int8_mode == "strict":
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.target_spec.supported_types = [tf.int8]
    else:
        converter.target_spec.supported_ops = [
            tf.lite.OpsSet.TFLITE_BUILTINS_INT8,
            tf.lite.OpsSet.TFLITE_BUILTINS
        ]
        converter.target_spec.supported_types = [tf.int8]

    try:
        print(f"Converting model={model_path}, rep={len(rep_data)}")
        tflite_model = converter.convert()
        return tflite_model, None
    except Exception as e:
        return None, str(e)

newQ = True
newC = True
mode = "strict"
rep_sizes = [2000, 3000, 4000, 5000]

results = []

'''for filename in os.listdir(MODEL_DIR):
    if not filename.endswith(".keras"):
        continue

    model_path = os.path.join(MODEL_DIR, filename)'''

import os

model_path = r"/content/drive/MyDrive/customized_normalized_onsets_APG_models/best_APG_onset_customized_normalized_non_quantized.keras"
OUTPUT_DIR = r"/content/drive/MyDrive/customized_normalized_onsets_APG_models"
base = os.path.splitext(os.path.basename(model_path))[0]

os.makedirs(OUTPUT_DIR, exist_ok=True)

results = []

for rep_size in rep_sizes:
    rep_data = X[:rep_size]

    out_name = f"{base}_rep{rep_size}.tflite"
    out_path = os.path.join(OUTPUT_DIR, out_name)

    if os.path.exists(out_path):
        print(f"[SKIP] {out_name} already exists. Skipping conversion.")
        results.append((out_name, "SKIPPED (exists)"))
        continue

    tflite_model, error = convert_model(
        model_path,
        rep_data,
        newQ,
        newC,
        mode
    )

    if tflite_model is not None:
        with open(out_path, "wb") as f:
            f.write(tflite_model)
        status = "SUCCESS"
    else:
        status = f"FAILED: {error}"

    results.append((out_name, status))
    print(f" → {status}")

print("\n=== SUMMARY ===")
for name, status in results:
    print(f"{name}: {status}")


Converting model=/content/drive/MyDrive/customized_normalized_onsets_APG_models/best_APG_onset_customized_normalized_non_quantized.keras, rep=2000
Saved artifact at '/tmp/tmptjnm0qm4'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 98, 1), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  137687099381648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137687099382608: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137687099381840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137687099382800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137687099383184: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137687099380688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137687099382416: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137687099383760: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13768709938203

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:854: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


 → SUCCESS
Converting model=/content/drive/MyDrive/customized_normalized_onsets_APG_models/best_APG_onset_customized_normalized_non_quantized.keras, rep=3000
Saved artifact at '/tmp/tmp6sw2r4vk'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 98, 1), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  137687447821712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137687447823056: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137687447822288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137687447823248: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137687447823632: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137687447821520: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137687447822864: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137687447824208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:854: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


 → SUCCESS
Converting model=/content/drive/MyDrive/customized_normalized_onsets_APG_models/best_APG_onset_customized_normalized_non_quantized.keras, rep=4000
Saved artifact at '/tmp/tmphpp74waq'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 98, 1), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  137687461092944: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137687461094288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137687461093520: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137687461094480: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137687461094864: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137687461092752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137687461094096: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137687461095440: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:854: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


 → SUCCESS
Converting model=/content/drive/MyDrive/customized_normalized_onsets_APG_models/best_APG_onset_customized_normalized_non_quantized.keras, rep=5000
Saved artifact at '/tmp/tmp6t5vbyfh'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 98, 1), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  137687447819600: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137687447817488: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137687447818640: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137687447818064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137687447817296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137687447819024: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137687447817680: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137687447816336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:854: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


 → SUCCESS

=== SUMMARY ===
best_APG_onset_customized_normalized_non_quantized_rep2000.tflite: SUCCESS
best_APG_onset_customized_normalized_non_quantized_rep3000.tflite: SUCCESS
best_APG_onset_customized_normalized_non_quantized_rep4000.tflite: SUCCESS
best_APG_onset_customized_normalized_non_quantized_rep5000.tflite: SUCCESS
